<a href="https://colab.research.google.com/github/mduberstein/llm_engineering/blob/michael-branch/Mike_Week_3_Day_5_Meeting_Minutes_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


## Again - please note: 2 important pro-tips for using Colab:

**Pro-tip 1:**

The top of every colab has some pip installs. You may receive errors from pip when you run this, such as:

> gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.

These pip compatibility errors can be safely ignored; and while it's tempting to try to fix them by changing version numbers, that will actually introduce real problems!

**Pro-tip 2:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [ ]:
!pip install -q --upgrade torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q requests bitsandbytes==0.46.0 transformers==4.48.3 accelerate==1.3.0 openai gradio

In [ ]:
# imports
import sys
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, TextIteratorStreamer, BitsAndBytesConfig
import torch
import gradio as gr
import threading

In [ ]:
# Constants

AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [ ]:
# New capability - connect this Colab to my Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3

# FOR PARTIAL DEBUGGING ONLY
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

In [ ]:
def _mount_drive_if_colab():
    """
    Try to mount Google Drive when running in Google Colab.
    A no-op outside Colab.
    """
    return_status = "Not running in Google Colab; skipping mount."
    try:
        if "google.colab" in sys.modules:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
            return_status = "Google Drive mounted at /content/drive"
            print(return_status)
        return return_status
    except Exception as e:
        return f"Drive mount failed: {e!r}"

# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [ ]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Sign in to OpenAI using Secrets in Colab

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [ ]:
# Use the Whisper OpenAI model to convert the Audio to Text
# If you'd prefer to use an Open Source model, class student Youssef has contributed an open source version
# which I've added to the bottom of this colab

# FOR PARTIAL DEBUGGING ONLY
audio_file = open(audio_filename, "rb")
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
# FOR PARTIAL DEBUGGING ONLY

system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
user_prompt = f"Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcription}"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
# Initialize Llama model and tokenizer

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)

In [ ]:
# FOR PARTIAL DEBUGGING ONLY
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)


In [ ]:
# FOR PARTIAL DEBUGGING ONLY
response = tokenizer.decode(outputs[0])

In [ ]:
# FOR PARTIAL DEBUGGING ONLY
display(Markdown(response))

In [ ]:
def transcribe_audio(audio_path: str) -> str:
    """
    Given a Google Drive path, return the full transcript as a single string.
    """
    # --- START OF YOUR IMPLEMENTATION ---
    if audio_path is None:
        return "Please upload an audio file."
    # Check file format
    if not str(audio_path).lower().endswith('.mp3'):
        return "Please upload an MP3 file."
    
    try:
        with open(audio_path, "rb") as audio_file:
            transcription = openai.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=audio_file,
            response_format="text"
        )
        return transcription
    except Exception as e:
        return f"Error during transcription: {str(e)}"

    return (
        f"[DEMO] Transcript produced by '{AUDIO_MODEL}' from: {drive_path}\n\n"
        f"{transcription}"       
    )
    # --- END OF YOUR IMPLEMENTATION ---


In [ ]:
def generate_minutes(transcript: str, max_tokens: int = 2000):
    system_message = (
        "You are an assistant that produces minutes of meetings from transcripts, "
        "with summary, key discussion points, takeaways and action items with owners, in markdown."
    )
    user_prompt = (
        f"Below is an extract transcript of a meeting. Please write minutes in markdown, "
        f"including a summary with attendees, location and date; discussion points; takeaways; "
        f"and action items with owners.\n{transcript}"
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    # Prepare input
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

      # Initialize the TextIteratorStreamer for streaming output
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        decode_kwargs={"skip_special_tokens": True}
    )
    thread = threading.Thread(
      target=model.generate,
      kwargs={"inputs": inputs, "max_new_tokens": max_tokens, "streamer": streamer}
    )
    
    thread.start()

    accumulated_response = []
    streamed_any = False
    try:
        for chunk in streamer:
            streamed_any = True
            filtered_chunk = chunk.replace("<|eot_id|>", "")  # Remove special tokens if present
            accumulated_response.append(filtered_chunk)
            yield "".join(accumulated_response)
    except Exception as e:
        print(f"Error occurred: {e}")
    finally:
        # Make sure the generation thread is cleaned up
        thread.join(timeout=0.1)
    
    # Fallback: if nothing streamed (some backends/models), do a blocking generate
    if not streamed_any:
        with torch.no_grad():
            out_ids = model.generate(inputs, max_new_tokens=2000)
            final_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
            yield final_text if final_text.strip() else "⚠️ Model returned no text."


In [ ]:
#Gradio UI setup
# ====== UI / APP =============================================================
with gr.Blocks(title="Week 3: Transcribe & Minutes Generator (Colab)") as demo:
    gr.Markdown("## Week 3 Homework — Transcription ➜ Minutes (Markdown)")

    with gr.Row():
        with gr.Column(scale=2):
            audio_path = gr.Textbox(
                label="Google Drive path to audio file",
                placeholder="/content/drive/MyDrive/llms/denver_extract.mp3",
                lines=2
            )
        with gr.Column(scale=1, min_width=240):
            mount_btn = gr.Button("Mount Google Drive (Colab)", variant="secondary")
            mount_status = gr.Markdown("", visible=True)
            clear_audio_btn = gr.Button("Clear Audio Path", variant="secondary")

    with gr.Row():
        transcribe_btn = gr.Button("Transcribe Audio", variant="primary")
        clear_transcript_btn = gr.Button("Clear Transcript", variant="secondary")

    transcript_tb = gr.Textbox(
        label="Transcript",
        placeholder="Transcript will appear here…",
        lines=18
    )

    gr.Markdown("---")

    with gr.Row():
        minutes_btn = gr.Button("Generate Minutes (Stream Markdown)", variant="primary")
        clear_minutes_btn = gr.Button("Clear Minutes", variant="secondary")

    minutes_md = gr.Markdown(
        value="",
        show_label=True,
        label="Minutes (Markdown)"
    )

    # ====== Wiring ===========================================================

    # Colab Drive mount button
    mount_btn.click(
        fn=_mount_drive_if_colab,
        inputs=None,
        outputs=mount_status,
        queue=False
    )

    # Transcribe handler -> writes to transcript textbox
    transcribe_btn.click(
        fn=transcribe_audio,
        inputs=[audio_path],
        outputs=[transcript_tb],
        queue=True  # set True if your ASR will be long-running
    )

    # Stream minutes from transcript -> Markdown
    minutes_btn.click(
        fn=generate_minutes,
        inputs=[transcript_tb],
        outputs=[minutes_md],
        queue=True  # needed for generator streaming
    )

    # Clear buttons
    clear_audio_btn.click(lambda: "", outputs=[audio_path], queue=False)
    clear_transcript_btn.click(lambda: "", outputs=[transcript_tb], queue=False)
    clear_minutes_btn.click(lambda: "", outputs=[minutes_md], queue=False)

    gr.Markdown(
        "ℹ️ **Notes:**\n"
        "- Set `AUDIO_MODEL` and `LLAMA` at the top of this file.\n"
        "- In Colab, use the **Mount Google Drive** button or mount manually."
    )


if __name__ == "__main__":
    # Colab-friendly launch: share=True gives a public URL
    # For very long tasks, you can also set server_port / server_name as needed.
    demo.queue().launch(share=True)
    # To run inside iframe in colab notebook
    # demo.queue().launch(share=True, inline=True)
    # To print error messages into the notebook
    # demo.queue().launch(share=True, inline=True, debug=True)
